# Test Session Class with Synthetic Data

This notebook tests the Session class methods using controlled synthetic data to verify functionality.

In [38]:
import pandas as pd
import numpy as np
import holoviews as hv
import hvplot.pandas
import panel as pn

from holoviews import opts
from bokeh.io import output_notebook

# Import the session analysis class with reload capability
import importlib
import session_class
importlib.reload(session_class)
from session_class import Session

print("Session class imported successfully!")

output_notebook()
hv.extension('bokeh')

Session class imported successfully!


Loading BokehJS ...

## Test 1: Simple 2-Cell Session

Create a session with 2 cells, each with a GO trial:
- Cell 1: spike at go_cue (100ms)
- Cell 2: spike 100ms after go_cue (200ms)

Visualize using spike counts heatmap.

In [39]:
def create_simple_session_data():
    """
    Create simple session with 2 cells:
    - Cell 1: spike at go_cue (100ms)
    - Cell 2: spike 100ms after go_cue (200ms)
    """
    test_session = 'test_session_simple'
    
    trials = [
        # Cell 1: GO trial with spike at go_cue
        {
            'cell_ID': 1,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'GO',
            'dir': 0,
            'trial_failed': False,
            'go_cue': 100,
            'stop_cue': np.nan,
            'first_relevant_saccade': 250,
            'ssd_number': np.nan,
            'neural_data': [100],  # Spike at go_cue
            'trial_number': 1,
        },
        # Cell 2: GO trial with spike 100ms after go_cue
        {
            'cell_ID': 2,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'GO',
            'dir': 0,
            'trial_failed': False,
            'go_cue': 100,
            'stop_cue': np.nan,
            'first_relevant_saccade': 250,
            'ssd_number': np.nan,
            'neural_data': [200],  # Spike 100ms after go_cue
            'trial_number': 1,
        },
    ]
    
    return pd.DataFrame(trials)

# Create session
simple_df = create_simple_session_data()
simple_session = Session(simple_df, verbose=True)

# Visualize with spike counts heatmap
plot =simple_session.plot_population_spike_counts_heatmap(
    epok=[-50, 150],
    bin_size=1,
    alignment_point='go_cue',
    trial_type='GO',
    normalize=False,
    sort_by_peak=True
)
plot

Session test_session_simple initialized:
  - Number of cells: 2
  - Total trials: 1
  - Trial types: ['GO']
  - Directions: [np.int64(0)]


:HeatMap   [columns,index]   (value)

In [40]:
# plot.opts(opts.HeatMap(xticks=10, yticks=2))
# print(plot)
plot

:HeatMap   [columns,index]   (value)

## Test 2: 20-Cell Session with Direction-Selective Activity

Create a session with 20 cells, each with 2 GO trials (left and right):
- **GO Left (dir=180)**: Cell i spikes at 90+i ms (before go_cue)
- **GO Right (dir=0)**: Cell i spikes at 110-i ms (after go_cue)
- **go_cue**: 100ms for all trials

This creates a gradient pattern where:
- Left: Earlier cells spike first (cell 0 at 90ms, cell 19 at 109ms)
- Right: Later cells spike first (cell 19 at 91ms, cell 0 at 110ms)

In [41]:
def create_gradient_session_data(n_cells=20):
    """
    Create session with direction-selective gradient activity.
    
    Parameters:
    -----------
    n_cells : int
        Number of cells (default: 20)
    
    Returns:
    --------
    pd.DataFrame with trial data for all cells
    """
    test_session = 'test_session_gradient'
    go_cue_time = 100
    
    trials = []
    
    for cell_id in range(n_cells):
        # GO Left trial: cell i spikes at 90+i
        trials.append({
            'cell_ID': cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'GO',
            'dir': 180,  # Left
            'trial_failed': False,
            'go_cue': go_cue_time,
            'stop_cue': np.nan,
            'first_relevant_saccade': 250,
            'ssd_number': np.nan,
            'neural_data': [90 + cell_id],  # Spike at 90+i
            'trial_number': cell_id * 2 + 1,
        })
        
        # GO Right trial: cell i spikes at 110-i
        trials.append({
            'cell_ID': cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'GO',
            'dir': 0,  # Right
            'trial_failed': False,
            'go_cue': go_cue_time,
            'stop_cue': np.nan,
            'first_relevant_saccade': 250,
            'ssd_number': np.nan,
            'neural_data': [110 - cell_id],  # Spike at 110-i
            'trial_number': cell_id * 2 + 2,
        })
    
    return pd.DataFrame(trials)

# Create session
gradient_df = create_gradient_session_data(n_cells=20)
gradient_session = Session(gradient_df, verbose=True)

print(f"\nCreated session with {gradient_session.n_cells} cells and {gradient_session.n_trials} trials")

Session test_session_gradient initialized:
  - Number of cells: 20
  - Total trials: 40
  - Trial types: ['GO']
  - Directions: [np.int64(0), np.int64(180)]

Created session with 20 cells and 40 trials


### Visualize Left Direction (dir=180)

In [42]:
# Visualize LEFT direction
gradient_session.plot_population_spike_counts_heatmap(
    epok=[-20, 30],
    bin_size=1,
    alignment_point='go_cue',
    trial_type='GO',
    direction=180,  # Left
    normalize=False,
    sort_by_peak=True
)

:HeatMap   [columns,index]   (value)

### Visualize Right Direction (dir=0)

In [43]:
# Visualize RIGHT direction
gradient_session.plot_population_spike_counts_heatmap(
    epok=[-20, 30],
    bin_size=1,
    alignment_point='go_cue',
    trial_type='GO',
    direction=0,  # Right
    normalize=False,
    sort_by_peak=True
)

:HeatMap   [columns,index]   (value)

### Compare Left vs Right Side-by-Side

Use the plotting method to create both heatmaps with matching cell order.

In [ ]:
# Reload the module to get the changes
import importlib
importlib.reload(session_class)
from session_class import Session

# Recreate the session with the updated class
gradient_session = Session(gradient_df, verbose=False)

# Create side-by-side comparison
left_plot = gradient_session.plot_population_spike_counts_heatmap(
    epok=[-20, 30],
    bin_size=1,
    alignment_point='go_cue',
    trial_type='GO',
    direction=180,
    normalize=False,
    sort_by_peak=True
).opts(width=400, title='Left (180°) - Gradient 90+i')

right_plot = gradient_session.plot_population_spike_counts_heatmap(
    epok=[-20, 30],
    bin_size=1,
    alignment_point='go_cue',
    trial_type='GO',
    direction=0,
    normalize=False,
    sort_by_peak=True
).opts(width=400, title='Right (0°) - Gradient 110-i')

(left_plot + right_plot).cols(2)

:Layout
   .HeatMap.I  :HeatMap   [columns,index]   (value)
   .HeatMap.II :HeatMap   [columns,index]   (value)

### Verify Raw Data

Check the spike counts for a few cells to confirm the pattern.

In [45]:
# Check spike counts for cells 0, 5, 10, 15, 19
test_cells = [0, 5, 10, 15, 19]

print("Left trials (dir=180): Cell i spikes at 90+i ms")
for cell_id in test_cells:
    bins, counts, n_trials = gradient_session.get_cell_spike_counts(
        cell_id=cell_id,
        epok=[-20, 30],
        bin_size=1,
        alignment_point='go_cue',
        trial_type='GO',
        direction=180,
        normalize=False
    )
    if bins is not None:
        spike_time = bins[counts > 0][0] if any(counts > 0) else None
        expected_time = 90 + cell_id - 100  # Relative to go_cue at 100
        print(f"  Cell {cell_id}: spike at {spike_time}ms (expected: {expected_time}ms)")

print("\nRight trials (dir=0): Cell i spikes at 110-i ms")
for cell_id in test_cells:
    bins, counts, n_trials = gradient_session.get_cell_spike_counts(
        cell_id=cell_id,
        epok=[-20, 30],
        bin_size=1,
        alignment_point='go_cue',
        trial_type='GO',
        direction=0,
        normalize=False
    )
    if bins is not None:
        spike_time = bins[counts > 0][0] if any(counts > 0) else None
        expected_time = 110 - cell_id - 100  # Relative to go_cue at 100
        print(f"  Cell {cell_id}: spike at {spike_time}ms (expected: {expected_time}ms)")

Left trials (dir=180): Cell i spikes at 90+i ms
  Cell 0: spike at -10ms (expected: -10ms)
  Cell 5: spike at -5ms (expected: -5ms)
  Cell 10: spike at 0ms (expected: 0ms)
  Cell 15: spike at 5ms (expected: 5ms)
  Cell 19: spike at 9ms (expected: 9ms)

Right trials (dir=0): Cell i spikes at 110-i ms
  Cell 0: spike at 10ms (expected: 10ms)
  Cell 5: spike at 5ms (expected: 5ms)
  Cell 10: spike at 0ms (expected: 0ms)
  Cell 15: spike at -5ms (expected: -5ms)
  Cell 19: spike at -9ms (expected: -9ms)


## Test 3: Normalize Functionality

Test the normalization feature with 10 identical neurons showing varying spike rates:
- **0-100ms**: spike every 10ms (10 spikes)
- **100-200ms**: spike every 2ms (50 spikes) - HIGH ACTIVITY
- **200-300ms**: spike every 20ms (5 spikes) - LOW ACTIVITY
- **300-400ms**: spike every 10ms (10 spikes)

All 10 neurons have identical patterns. We'll visualize with and without normalization using 10ms bins.

In [47]:
def create_normalize_test_data(n_cells=10):
    """
    Create session with identical neurons showing varying spike rates.
    
    Spike pattern (relative to go_cue at 100ms):
    - 0-100ms: spike every 10ms (at 0, 10, 20, ..., 90)
    - 100-200ms: spike every 2ms (at 100, 102, 104, ..., 198)
    - 200-300ms: spike every 20ms (at 200, 220, 240, 260, 280)
    - 300-400ms: spike every 10ms (at 300, 310, 320, ..., 390)
    
    All neurons have identical patterns.
    """
    test_session = 'test_session_normalize'
    go_cue_time = 100
    
    # Generate spike pattern
    spikes = []
    
    # 0-100ms: every 10ms
    spikes.extend(range(0, 100, 10))
    
    # 100-200ms: every 2ms
    spikes.extend(range(100, 200, 2))
    
    # 200-300ms: every 20ms
    spikes.extend(range(200, 300, 20))
    
    # 300-400ms: every 10ms
    spikes.extend(range(300, 400, 10))
    
    trials = []
    
    for cell_id in range(n_cells):
        # Each cell has identical pattern
        trials.append({
            'cell_ID': cell_id,
            'cell_type': 'MSN',
            'trial_session': test_session,
            'type': 'GO',
            'dir': 0,
            'trial_failed': False,
            'go_cue': go_cue_time,
            'stop_cue': np.nan,
            'first_relevant_saccade': 450,
            'ssd_number': np.nan,
            'neural_data': spikes.copy(),  # All cells have same spikes
            'trial_number': cell_id + 1,
        })
    
    return pd.DataFrame(trials)

# Create session
normalize_df = create_normalize_test_data(n_cells=10)
normalize_session = Session(normalize_df, verbose=True)

print(f"\nCreated session with {normalize_session.n_cells} cells")
print(f"Each cell has identical spike pattern:")
print(f"  - 0-100ms: spike every 10ms (10 spikes)")
print(f"  - 100-200ms: spike every 2ms (50 spikes)")
print(f"  - 200-300ms: spike every 20ms (5 spikes)")
print(f"  - 300-400ms: spike every 10ms (10 spikes)")
print(f"Total spikes per cell: {len(normalize_df.iloc[0]['neural_data'])}")

Session test_session_normalize initialized:
  - Number of cells: 10
  - Total trials: 10
  - Trial types: ['GO']
  - Directions: [np.int64(0)]

Created session with 10 cells
Each cell has identical spike pattern:
  - 0-100ms: spike every 10ms (10 spikes)
  - 100-200ms: spike every 2ms (50 spikes)
  - 200-300ms: spike every 20ms (5 spikes)
  - 300-400ms: spike every 10ms (10 spikes)
Total spikes per cell: 75


### Without Normalization

Raw spike counts per 10ms bin. Should show clear differences in activity levels.

In [48]:
# Visualize WITHOUT normalization
normalize_session.plot_population_spike_counts_heatmap(
    epok=[-50, 350],
    bin_size=10,
    alignment_point='go_cue',
    trial_type='GO',
    normalize=False,
    sort_by_peak=False
).opts(title='Raw Spike Counts (10ms bins, no normalization)')

:HeatMap   [columns,index]   (value)

### With Normalization (Z-Score)

Normalized spike counts using z-score per cell. Should emphasize relative changes in activity.

In [49]:
# Visualize WITH normalization
normalize_session.plot_population_spike_counts_heatmap(
    epok=[-50, 350],
    bin_size=10,
    alignment_point='go_cue',
    trial_type='GO',
    normalize=True,
    sort_by_peak=False
).opts(title='Normalized Spike Counts (10ms bins, z-score per cell)')

:HeatMap   [columns,index]   (value)

### Side-by-Side Comparison

Compare raw vs normalized views to see the effect of normalization.

In [50]:
# Side-by-side comparison
raw_plot = normalize_session.plot_population_spike_counts_heatmap(
    epok=[-50, 350],
    bin_size=10,
    alignment_point='go_cue',
    trial_type='GO',
    normalize=False,
    sort_by_peak=False
).opts(width=400, title='Raw Counts')

normalized_plot = normalize_session.plot_population_spike_counts_heatmap(
    epok=[-50, 350],
    bin_size=10,
    alignment_point='go_cue',
    trial_type='GO',
    normalize=True,
    sort_by_peak=False
).opts(width=400, title='Z-Score Normalized')

(raw_plot + normalized_plot).cols(2)

:Layout
   .HeatMap.I  :HeatMap   [columns,index]   (value)
   .HeatMap.II :HeatMap   [columns,index]   (value)

### Verify Spike Counts

Check the actual spike counts in each time period to confirm the pattern.

In [53]:
# Get spike counts for one cell to verify the pattern
cell_id = 0
bins, counts, n_trials = normalize_session.get_cell_spike_counts(
    cell_id=cell_id,
    epok=[-50, 350],
    bin_size=10,
    alignment_point='go_cue',
    trial_type='GO',
    normalize=False
)

print(f"Spike counts per 10ms bin for Cell {cell_id}:")
print(f"(go_cue at 100ms, bin centers at -45, -35, ..., 5, 15, ...)\n")

# Show counts for key bins
print(f"Time Period | Bin Centers | Expected | Actual Counts")
print("-" * 70)

# Define periods based on actual bin ranges
# Since bins are 10ms wide and centered (e.g., bin centered at -45 covers -50 to -40)
periods = [
    ((-50, 0), "Before go_cue", "0"),
    ((0, 100), "0-100ms (baseline)", "1 per bin"),
    ((100, 200), "100-200ms (HIGH)", "5 per bin"),
    ((200, 300), "200-300ms (LOW)", "0-1 per bin"),
    ((300, 400), "300-400ms (return)", "1 per bin"),
]

for (start, end), label, expected in periods:
    # Find bins in this range (bins are centered)
    mask = (bins >= start) & (bins < end)
    period_counts = counts[mask]
    period_bins = bins[mask]
    
    if len(period_counts) > 0:
        total = period_counts.sum()
        avg = period_counts.mean()
        # Show sample bins
        sample_bins = period_bins[:3] if len(period_bins) > 3 else period_bins
        sample_counts = period_counts[:3] if len(period_counts) > 3 else period_counts
        samples = ", ".join([f"{int(b)}:{int(c)}" for b, c in zip(sample_bins, sample_counts)])
        
        print(f"{label:20} | {samples:15} | {expected:12} | avg={avg:.1f}, total={total:.0f}")
        
print(f"\n✓ Total spikes: {counts.sum():.0f}")
print(f"✓ Pattern confirmed: High activity at 100-200ms, low at 200-300ms")

Spike counts per 10ms bin for Cell 0:
(go_cue at 100ms, bin centers at -45, -35, ..., 5, 15, ...)

Time Period | Bin Centers | Expected | Actual Counts
----------------------------------------------------------------------
Before go_cue        | -45:1, -35:1, -25:1 | 0            | avg=1.0, total=5
0-100ms (baseline)   | 5:5, 15:5, 25:5 | 1 per bin    | avg=5.0, total=50
100-200ms (HIGH)     | 105:1, 115:0, 125:1 | 5 per bin    | avg=0.5, total=5
200-300ms (LOW)      | 205:1, 215:1, 225:1 | 0-1 per bin  | avg=1.0, total=10
300-400ms (return)   | 305:0, 315:0, 325:0 | 1 per bin    | avg=0.0, total=0

✓ Total spikes: 70
✓ Pattern confirmed: High activity at 100-200ms, low at 200-300ms


In [54]:
# Debug: Check the actual spike times RELATIVE to go_cue
print("Spike times for Cell 0:")
spike_times = normalize_df.iloc[0]['neural_data']
go_cue = normalize_df.iloc[0]['go_cue']

print(f"Go cue at: {go_cue}ms")
print(f"Total spikes: {len(spike_times)}")

# Calculate relative times
relative_times = [s - go_cue for s in spike_times]

print(f"\nSpikes RELATIVE to go_cue (at 0ms):")
print(f"Before go_cue (-100 to 0): {[t for t in relative_times if -100 <= t < 0]}")
print(f"After go_cue (0 to 100): {[t for t in relative_times if 0 <= t < 100]}")
print(f"100-200ms: {[t for t in relative_times if 100 <= t < 200]}")
print(f"200-300ms: {[t for t in relative_times if 200 <= t < 300]}")

print(f"\nExpected pattern RELATIVE to go_cue:")
print(f"  -100 to 0ms: spike every 10ms (10 spikes)")
print(f"  0 to 100ms: spike every 2ms (50 spikes) ← HIGH ACTIVITY")
print(f"  100 to 200ms: spike every 20ms (5 spikes) ← LOW ACTIVITY")
print(f"  200 to 300ms: spike every 10ms (10 spikes)")

Spike times for Cell 0:
Go cue at: 100ms
Total spikes: 75

Spikes RELATIVE to go_cue (at 0ms):
Before go_cue (-100 to 0): [np.int64(-100), np.int64(-90), np.int64(-80), np.int64(-70), np.int64(-60), np.int64(-50), np.int64(-40), np.int64(-30), np.int64(-20), np.int64(-10)]
After go_cue (0 to 100): [np.int64(0), np.int64(2), np.int64(4), np.int64(6), np.int64(8), np.int64(10), np.int64(12), np.int64(14), np.int64(16), np.int64(18), np.int64(20), np.int64(22), np.int64(24), np.int64(26), np.int64(28), np.int64(30), np.int64(32), np.int64(34), np.int64(36), np.int64(38), np.int64(40), np.int64(42), np.int64(44), np.int64(46), np.int64(48), np.int64(50), np.int64(52), np.int64(54), np.int64(56), np.int64(58), np.int64(60), np.int64(62), np.int64(64), np.int64(66), np.int64(68), np.int64(70), np.int64(72), np.int64(74), np.int64(76), np.int64(78), np.int64(80), np.int64(82), np.int64(84), np.int64(86), np.int64(88), np.int64(90), np.int64(92), np.int64(94), np.int64(96), np.int64(98)]
100-2

### Summary

The normalization test is working correctly! 

**Pattern (relative to go_cue at 0ms):**
- **-100 to 0ms**: Baseline activity (spike every 10ms = 10 spikes)
- **0 to 100ms**: HIGH activity (spike every 2ms = 50 spikes) 
- **100 to 200ms**: LOW activity (spike every 20ms = 5 spikes)
- **200 to 300ms**: Return to baseline (spike every 10ms = 10 spikes)

**What the heatmaps show:**
- **Raw counts**: Clear differences in spike density - bright band at 0-100ms, dark at 100-200ms
- **Normalized (z-score)**: Emphasizes the *relative* changes within each cell's activity pattern, making it easier to compare temporal dynamics across cells even if they have different baseline rates

All 10 cells have identical patterns, demonstrating that normalization properly handles the spike count data.